# 03d - Hyperparameter Tuning
RandomizedSearchCV pro vsechny baseline modely. SMOTE je soucasti pipeline, aby nedoslo k data leakage pri cross-validaci.
Vystupy: `models/tuned/*.pkl`, `models/best_model.pkl`, `models/best_model_metadata.json`

In [1]:
import joblib
import warnings
import os
import json
from datetime import datetime
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV

warnings.filterwarnings('ignore')

X_train_prep, X_test_prep, y_train, y_test = joblib.load('../data/processed/split_data.pkl')

import matplotlib as _mpl

# Paul Tol bright palette — colorblind-safe, perceptually uniform
_PALETTE = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377', '#BBBBBB']

_mpl.rcParams.update({
    # Color cycle
    'axes.prop_cycle':       _mpl.cycler(color=_PALETTE),

    # Backgrounds
    'figure.facecolor':      'white',
    'axes.facecolor':        '#F8F9FA',   # subtle off-white — "card" feel

    # Spines
    'axes.edgecolor':        '#DEE2E6',
    'axes.linewidth':        0.9,
    'axes.spines.top':       False,
    'axes.spines.right':     False,

    # Grid — solid, very light
    'axes.grid':             True,
    'axes.axisbelow':        True,
    'grid.color':            '#E9ECEF',
    'grid.linestyle':        '-',
    'grid.linewidth':        0.8,
    'grid.alpha':            1.0,

    # Typography
    'font.family':           'sans-serif',
    'font.sans-serif':       ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'axes.titlesize':        14,
    'axes.titleweight':      'bold',
    'axes.titlepad':         14,
    'axes.labelsize':        12,
    'axes.labelweight':      'regular',
    'xtick.labelsize':       10,
    'ytick.labelsize':       10,
    'legend.fontsize':       10,
    'legend.title_fontsize': 11,
    'figure.titlesize':      16,
    'figure.titleweight':    'bold',

    # Text colors — near-black for contrast
    'text.color':            '#212529',
    'axes.labelcolor':       '#495057',
    'xtick.color':           '#495057',
    'ytick.color':           '#495057',

    # Ticks
    'xtick.direction':       'out',
    'ytick.direction':       'out',
    'xtick.major.size':      4,
    'ytick.major.size':      4,
    'xtick.minor.visible':   False,
    'ytick.minor.visible':   False,
    'xtick.major.pad':       5,
    'ytick.major.pad':       5,

    # Lines & markers
    'lines.linewidth':       2.0,
    'lines.markersize':      7,
    'patch.linewidth':       0.6,

    # Legend
    'legend.frameon':        True,
    'legend.framealpha':     0.92,
    'legend.edgecolor':      '#DEE2E6',
    'legend.fancybox':       True,
    'legend.borderpad':      0.6,

    # Figure / saving
    'figure.dpi':            100,
    'figure.figsize':        [8, 5],
    'savefig.dpi':           150,
    'savefig.bbox':          'tight',
    'savefig.facecolor':     'white',
    'savefig.edgecolor':     'none',
})


## Definice modelu a param gridu

In [2]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
}

param_grids = {
    'Logistic Regression': {
        'model__C':            [0.01, 0.1, 1, 10, 100],
        'model__penalty':      ['l2'],
        'model__class_weight': [None, 'balanced']
    },
    'Random Forest': {
        'model__n_estimators':      [100, 200, 300],
        'model__max_depth':         [5, 10, 15, None],
        'model__min_samples_split': [2, 5, 10]
    },
    'Gradient Boosting': {
        'model__n_estimators':  [100, 200, 300],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__max_depth':     [3, 4, 5]
    },
    'Decision Tree': {
        'model__max_depth':         [3, 5, 7, 10, None],
        'model__min_samples_split': [2, 5, 10, 20],
        'model__min_samples_leaf':  [1, 2, 5],
        'model__class_weight':      [None, 'balanced'],
    },
}


## Tuning

In [3]:
print("Spoustim ladeni hyperparametru (Hyperparameter Tuning)...\n")

best_tuned_models  = {}
overall_best_model  = None
overall_best_score  = 0
overall_best_params = {}
winning_name        = ""
all_cv_results      = {}

for name, model in models.items():
    print(f"Ladim: {name}")

    # SMOTE v pipeline -> zadny data leakage pri CV
    tuning_pipeline = ImbPipeline(steps=[
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    search = RandomizedSearchCV(
        tuning_pipeline,
        param_distributions=param_grids[name],
        n_iter=10,
        scoring='roc_auc',
        cv=3,           # 3-fold cross validation
        random_state=42,
        n_jobs=-1       # Vyuzije vsechna jadra procesoru
    )

    search.fit(X_train_prep, y_train)

    best_tuned_models[name] = search.best_estimator_
    print(f"   Nejlepsi parametry: {search.best_params_}")
    print(f"   Nejlepsi ROC-AUC : {search.best_score_:.4f}\n")

    if search.best_score_ > overall_best_score:
        overall_best_score  = search.best_score_
        overall_best_model  = search.best_estimator_
        overall_best_params = search.best_params_
        winning_name        = name

    # --- Zachytit vsechny vyzkoušené kombinace ---
    import pandas as pd
    cv_df = pd.DataFrame(search.cv_results_)
    param_cols = [c for c in cv_df.columns if c.startswith('param_')]
    cv_df_clean = cv_df[param_cols + ['mean_test_score', 'std_test_score', 'rank_test_score']].copy()
    cv_df_clean.columns = [c.replace('param_model__', '') for c in cv_df_clean.columns]
    cv_df_clean = cv_df_clean.sort_values('rank_test_score').reset_index(drop=True)
    cv_df_clean['mean_test_score'] = cv_df_clean['mean_test_score'].round(4)
    cv_df_clean['std_test_score']  = cv_df_clean['std_test_score'].round(4)
    all_cv_results[name] = cv_df_clean

    os.makedirs('../results', exist_ok=True)
    safe_name_csv = name.lower().replace(' ', '_')
    cv_df_clean.to_csv(f'../results/hp_search_{safe_name_csv}.csv', index=False)
    print(f"   Vsechny kombinace ulozeny: results/hp_search_{safe_name_csv}.csv")
    print(cv_df_clean.to_string(index=False))
    print()

print("==================================================")
print(f"ABSOLUTNI VITEZ: {winning_name} (ROC-AUC: {overall_best_score:.4f})")
print("==================================================")


Spoustim ladeni hyperparametru (Hyperparameter Tuning)...

Ladim: Logistic Regression


   Nejlepsi parametry: {'model__penalty': 'l2', 'model__class_weight': None, 'model__C': 0.01}
   Nejlepsi ROC-AUC : 0.8000

   Vsechny kombinace ulozeny: results/hp_search_logistic_regression.csv
penalty class_weight      C  mean_test_score  std_test_score  rank_test_score
     l2          NaN   0.01           0.8000          0.0033                1
     l2     balanced   0.01           0.8000          0.0033                1
     l2          NaN   0.10           0.7957          0.0030                3
     l2     balanced   0.10           0.7957          0.0030                3
     l2          NaN   1.00           0.7951          0.0029                5
     l2     balanced   1.00           0.7951          0.0029                5
     l2          NaN  10.00           0.7950          0.0029                7
     l2     balanced  10.00           0.7950          0.0029                7
     l2          NaN 100.00           0.7950          0.0029                9
     l2     balanced 10

   Nejlepsi parametry: {'model__n_estimators': 200, 'model__min_samples_split': 5, 'model__max_depth': 10}
   Nejlepsi ROC-AUC : 0.7926

   Vsechny kombinace ulozeny: results/hp_search_random_forest.csv
 n_estimators  min_samples_split max_depth  mean_test_score  std_test_score  rank_test_score
          200                  5        10           0.7926          0.0057                1
          300                 10        10           0.7924          0.0054                2
          100                  5        10           0.7923          0.0059                3
          200                 10        10           0.7923          0.0053                4
          300                 10        15           0.7907          0.0049                5
          300                 10      None           0.7901          0.0055                6
          100                  5        15           0.7884          0.0052                7
          200                  5      None           

   Nejlepsi parametry: {'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.05}
   Nejlepsi ROC-AUC : 0.7964

   Vsechny kombinace ulozeny: results/hp_search_gradient_boosting.csv
 n_estimators  max_depth  learning_rate  mean_test_score  std_test_score  rank_test_score
          300          3           0.05           0.7964          0.0052                1
          100          3           0.05           0.7961          0.0053                2
          100          4           0.05           0.7948          0.0056                3
          300          5           0.01           0.7945          0.0053                4
          200          4           0.05           0.7945          0.0060                5
          100          4           0.10           0.7925          0.0060                6
          200          5           0.05           0.7909          0.0059                7
          100          5           0.10           0.7898          0.0063   

   Nejlepsi parametry: {'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 3, 'model__class_weight': None}
   Nejlepsi ROC-AUC : 0.7761

   Vsechny kombinace ulozeny: results/hp_search_decision_tree.csv
 min_samples_split  min_samples_leaf max_depth class_weight  mean_test_score  std_test_score  rank_test_score
                 2                 2         3          NaN           0.7761          0.0019                1
                 2                 2         3     balanced           0.7761          0.0019                1
                10                 5         3          NaN           0.7761          0.0019                1
                 5                 1         5     balanced           0.7693          0.0032                4
                10                 1         7          NaN           0.7464          0.0062                5
                20                 5        10          NaN           0.7165          0.0006               

## Ulozeni modelu

In [4]:
os.makedirs('../models/tuned', exist_ok=True)

# Vsechny vylazene modely
for name, estimator in best_tuned_models.items():
    safe_name = name.lower().replace(' ', '_')
    joblib.dump(estimator, f'../models/tuned/{safe_name}_tuned.pkl')
    print(f"  models/tuned/{safe_name}_tuned.pkl")

# Vitez zvlast - jednoznacny vstup pro downstream notebooky (XAI, serving)
joblib.dump(overall_best_model, '../models/best_model.pkl')
print("\nVitezny model ulozen jako models/best_model.pkl")

# Metadata
metadata = {
    "winning_model":   winning_name,
    "roc_auc_cv":      overall_best_score,
    "best_params":     overall_best_params,
    "trained_on":      datetime.now().isoformat(),
    "source_notebook": "03d-hyperparameter-tuning.ipynb"
}
with open('../models/best_model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Metadata ulozena jako models/best_model_metadata.json")
print(json.dumps(metadata, indent=2, ensure_ascii=False))


  models/tuned/logistic_regression_tuned.pkl
  models/tuned/random_forest_tuned.pkl
  models/tuned/gradient_boosting_tuned.pkl
  models/tuned/decision_tree_tuned.pkl

Vitezny model ulozen jako models/best_model.pkl
Metadata ulozena jako models/best_model_metadata.json
{
  "winning_model": "Logistic Regression",
  "roc_auc_cv": 0.800011634284053,
  "best_params": {
    "model__penalty": "l2",
    "model__class_weight": null,
    "model__C": 0.01
  },
  "trained_on": "2026-04-20T21:45:49.661969",
  "source_notebook": "03d-hyperparameter-tuning.ipynb"
}
